# 全连接 vs 卷积自编码器：CIFAR-10 受控对比实验

用一组受控对照实验回答：**把自编码器从全连接（MLP）换成卷积（CNN），哪些改进真实发生，哪些"教科书预期"其实站不住脚？**

> 一句话结论：卷积架构用 **1/10 的参数量**换来 **+1.25 dB PSNR** 与 **6 倍收敛加速**；
> 但"更强的平移鲁棒性"与"更具语义的隐空间"这两个常见预期**均未成立**——
> 而解释"为什么没发生"恰是本实验最有价值的部分。

## 实验设计

| | Classic AE（全连接） | CNN AE（卷积） |
|---|---|---|
| 结构 | 3072→1024→256→**128**→256→1024→3072 | Conv/DeConv 栈，瓶颈 **128** |
| 瓶颈维度 | 128 | 128 |
| 损失 / 优化器 | MSE / Adam | MSE / Adam（超参完全一致） |
| 数据 / 训练 | CIFAR-10，50 epoch | CIFAR-10，50 epoch |
| 参数量 | 6.89M | **0.71M** |

两模型仅架构不同，其余完全一致——所有差异均可归因于架构本身。

## 核心结果

| 指标 | Classic AE | CNN AE | 差异 |
|---|---|---|---|
| Test MSE ↓ | 0.0094 | 0.0071 | **−24%** |
| PSNR ↑ | 20.83 dB | 22.08 dB | **+1.25 dB** |
| 收敛至 MSE≈0.0105 | ~30 epoch | **~5 epoch** | **6× 加速** |
| CPU 推理延迟（ms/张） | **0.082** | 0.126 | Classic 反而快 ~35% |
| 线性探针分类准确率 | 39.3% | 38.6% | ≈ 持平 |
| 隐空间有效维度（PR/128） | 56.8 | **71.5** | CNN 利用更均匀 |

## 三个假设的记分卡

### H1 参数效率 —— ✅ 成立

CNN AE 以 **1/10 的参数量**（0.71M vs 6.89M）取得更优重构（PSNR +1.25 dB，MSE −24%），并带来 **6 倍收敛加速**（5 vs 30 epoch 达到同水平）——权重共享使有效搜索空间大幅缩小，优化信号更有效。分类别评估显示**纹理复杂的动物类受益最大**（cat +1.46、dog +1.43 dB），frog 最小（+1.00 dB）。

### H2 平移鲁棒性 —— ⚠️ 预期修正

| 平移量 | 0px | 2px | 4px | 8px | 16px |
|---|---|---|---|---|---|
| Classic AE | 20.83 | 19.92 | 19.71 | 19.48 | 19.33 |
| CNN AE | 22.08 | 21.12 | 20.96 | 20.80 | 20.66 |

两条曲线**近乎平行**：CNN 的优势是恒定的 ~1.3 dB 质量偏移，而非随位移扩大的鲁棒性优势；16px 时两者降幅几乎相同（−1.50 vs −1.42 dB），全连接模型并未出现预期的"位置硬绑定崩塌"。归因：

1. CIFAR-10 中物体位置高度随机，连全连接层也被迫学出位置容忍；
2. Classic AE 的模糊重构本身对位移不敏感——其表观稳健是低重建质量的副产品，而非结构理解。

### H3 隐空间语义性 —— ❌ 证伪 → 更有价值的替代发现

t-SNE 显示两者隐空间均仅呈**弱类别聚类**（主导轴是背景色调而非语义）；线性探针准确率几乎一致（39.3% vs 38.6%，随机基线 10%）。归因：MSE 重构目标不奖励类别分离，且 128 维瓶颈足够宽松，模型无需抽象类别级共享因子即可完成重构。

由此得到本实验的核心发现：**重构质量与表征判别力是解耦的**——架构改进显著改善了重构与隐空间利用率（PR 56.8→71.5），却未增加线性可提取的类别信息。这与自监督学习文献中"目标函数比架构更决定表征质量"的结论一致。

### 附：效率的细微之处

CNN 的优势体现在**参数量（−90%）与显存**，但 CPU 推理延迟反而更高（0.126 vs 0.082 ms/张）——两个大矩阵乘在 CPU 上非常高效。"参数少 = 推理快"并不总成立，需分硬件、分算子讨论。

## 可视化结果

**重构对比**（上：原图；中：Classic AE；下：CNN AE）——CNN 保留了船体轮廓、车门剪影等中层结构：

![并排对比](results_compare/side_by_side.png)

**训练曲线**——CNN 约 5 epoch 达到 Classic 30 epoch 的水平：

![Classic 训练曲线](results_classic/loss_curve.png)
![CNN 训练曲线](results_conv/loss_curve.png)

**平移扫描**——两曲线平行，无大位移崩塌：

![平移扫描](results_compare/shift_curve.png)

**t-SNE**——两者均弱聚类：

![t-SNE](results_compare/latent_tsne.png)

**分类别 PSNR**（动物类差距最大）：

![分类别](results_full/per_class_psnr.png)

**误差热力图**——Classic 误差均匀偏大，CNN 误差集中于纹理区：

![Classic 误差](results_full/error_heatmap_classic.png)
![CNN 误差](results_full/error_heatmap_cnn.png)

**隐空间插值**（cat→ship）与**维度利用率**：

![插值](results_full/latent_interp.png)
![维度利用率](results_full/dim_utilization.png)

**线性探针混淆矩阵**：

![混淆矩阵](results_full/probe_confusion.png)

## 复现



# MLP vs. Convolutional Autoencoder: A Controlled Comparison on CIFAR-10

A controlled experiment answering one question: **when we swap a fully-connected
(MLP) autoencoder for a convolutional one, which improvements are real — and
which "textbook expectations" don't actually hold up?**

> **TL;DR:** The convolutional architecture delivers **+1.25 dB PSNR** and a
> **6x convergence speedup** with **1/10th the parameters**. But two common
> expectations — "better translation robustness" and "a more semantic latent
> space" — **both failed to materialize**. Explaining *why* they didn't is the
> most valuable part of this experiment.

## Experimental Design

| | Classic AE (MLP) | CNN AE (Convolutional) |
|---|---|---|
| Architecture | 3072→1024→256→**128**→256→1024→3072 | Conv/DeConv stack, bottleneck **128** |
| Bottleneck dim | 128 | 128 |
| Loss / Optimizer | MSE / Adam | MSE / Adam (identical hyperparameters) |
| Data / Training | CIFAR-10, 50 epochs | CIFAR-10, 50 epochs |
| Parameters | 6.89M | **0.71M** |

The two models differ *only* in architecture; everything else is identical, so
all differences are attributable to the architecture itself.

## Core Results

| Metric | Classic AE | CNN AE | Delta |
|---|---|---|---|
| Test MSE (lower better) | 0.0094 | 0.0071 | **−24%** |
| PSNR (higher better) | 20.83 dB | 22.08 dB | **+1.25 dB** |
| Epochs to reach MSE ≈ 0.0105 | ~30 | **~5** | **6x faster** |
| CPU inference latency (ms/img) | **0.082** | 0.126 | Classic is ~35% faster |
| Linear-probe classification accuracy | 39.3% | 38.6% | ≈ tie |
| Effective latent dims (participation ratio /128) | 56.8 | **71.5** | CNN uses its latent space more evenly |

## Hypothesis Scorecard

### H1: Parameter Efficiency — ✅ Confirmed

The CNN AE achieves better reconstruction with **1/10th the parameters**
(0.71M vs 6.89M): PSNR +1.25 dB, MSE −24%, plus a **6x convergence speedup**
(~5 vs ~30 epochs to the same level). Weight sharing shrinks the effective
hypothesis space and makes the optimization signal far more effective.
Per-class analysis shows the **largest gains on texture-heavy animal classes**
(cat +1.46 dB, dog +1.43 dB) and the smallest on frog (+1.00 dB).

### H2: Translation Robustness — ⚠️ Expectation Revised

| Shift (px, wrap-around) | 0 | 2 | 4 | 8 | 16 |
|---|---|---|---|---|---|
| Classic AE (dB) | 20.83 | 19.92 | 19.71 | 19.48 | 19.33 |
| CNN AE (dB) | 22.08 | 21.12 | 20.96 | 20.80 | 20.66 |

The two curves are **nearly parallel**: the CNN's advantage is a constant
~1.3 dB quality offset, not a robustness advantage that grows with shift. At
16px the degradation is almost identical (−1.50 vs −1.42 dB), and the expected
"positional hard-binding collapse" of the MLP never appears. Two explanations:

1. Object positions in CIFAR-10 are highly random, so even the fully-connected
   layer is forced to learn positional tolerance;
2. The Classic AE's blurry reconstructions are inherently insensitive to
   input shifts — its apparent robustness is a *side effect of poor
   reconstruction quality*, not structural understanding.

### H3: Latent Space Semantics — ❌ Falsified → A More Interesting Finding

t-SNE shows only **weak class clustering** in both models (the dominant axes
are background color layout, not semantics), and linear-probe accuracies are
statistically tied (39.3% vs 38.6%; random baseline 10%). Why: the MSE
reconstruction objective does not reward class separation, and a 128-dim
bottleneck is roomy enough to reconstruct without abstracting class-level
shared factors.

This yields the experiment's central finding: **reconstruction quality and
representation discriminability are decoupled in this setting.** The
architectural upgrade substantially improved reconstruction and latent-space
utilization (PR 56.8 → 71.5) yet added no linearly extractable class
information — consistent with the self-supervised learning literature's
conclusion that the *objective function*, more than the architecture,
determines representation quality.

### Aside: The Nuance in "Efficiency"

The CNN's advantage shows up in **parameters (−90%) and memory**, but CPU
inference latency is actually *higher* (0.126 vs 0.082 ms/img) — two large
matrix multiplications are very fast on CPU. "Fewer parameters = faster
inference" does not always hold; it must be discussed per hardware and per
operator.

## Visual Results

**Reconstructions** (top: original; middle: Classic AE; bottom: CNN AE) — the
CNN preserves mid-level structure like hull outlines and car silhouettes:

![Side-by-side comparison](results_compare/side_by_side.png)

**Training curves** — the CNN reaches in ~5 epochs what the Classic AE needs
~30 epochs for:

![Classic training curve](results_classic/loss_curve.png)
![CNN training curve](results_conv/loss_curve.png)

**Translation sweep** — parallel curves, no collapse at large shifts:

![Translation sweep](results_compare/shift_curve.png)

**t-SNE** — weak clustering in both latent spaces:

![t-SNE](results_compare/latent_tsne.png)

**Per-class PSNR** (animal classes benefit most):

![Per-class PSNR](results_full/per_class_psnr.png)

**Error heatmaps** — Classic: uniformly larger error; CNN: error concentrated
in textured regions:

![Classic error heatmap](results_full/error_heatmap_classic.png)
![CNN error heatmap](results_full/error_heatmap_cnn.png)

**Latent interpolation** (cat → ship) and **dimension utilization**:

![Latent interpolation](results_full/latent_interp.png)
![Dimension utilization](results_full/dim_utilization.png)

**Linear-probe confusion matrices**:

![Confusion matrices](results_full/probe_confusion.png)

## Reproduction

